# 11 — Urban form and thermal accessibility diagnostics

This notebook explores why accessibility losses differ across sampled origins.

Two spatial scales are tested:

1. **local 250 m buffers** around each accessibility origin;
2. the **normally reachable walking network** within the 600 s walking-time budget.

These are exploratory mechanism diagnostics, not causal models.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import rasterio

from rasterio.mask import mask
from shapely.geometry import mapping

ACCESS_DIR = OUTPUTS / "accessibility"
SOLWEIG_DIR = OUTPUTS / "solweig" / "summary"

access = pd.read_csv(
    ACCESS_DIR / "origin_accessibility_lambda_005.csv"
)
access["origin"] = access["origin"].astype(str)

G = ox.load_graphml(
    DATA_INTERIM / "sevilla_walk_thermal.graphml"
)

numeric_fields = [
    "length",
    "walk_time_s",
    "utci_max",
    "heat_excess",
    "thermal_cost_s",
    "cost_ratio",
]

for _, _, _, data in G.edges(keys=True, data=True):
    for field in numeric_fields:
        if field in data:
            try:
                data[field] = float(data[field])
            except (TypeError, ValueError):
                data[field] = np.nan

## 1. Recover origin geometries and create 250 m buffers

In [ ]:
nodes, _ = ox.graph_to_gdfs(G)
nodes = nodes.reset_index()
nodes["osmid"] = nodes["osmid"].astype(str)

origins = (
    nodes[nodes["osmid"].isin(access["origin"])]
    .merge(
        access,
        left_on="osmid",
        right_on="origin",
        how="inner",
    )
    .to_crs(25830)
)

BUFFER_M = 250

buffers = origins[
    ["origin", "access_loss_pct", "geometry"]
].copy()

buffers["geometry"] = buffers.geometry.buffer(BUFFER_M)

print("Origins:", len(origins))

## 2. Building metrics

In [ ]:
buildings = gpd.read_file(
    DATA_INTERIM / "sevilla_buildings_voxcity.gpkg"
).to_crs(25830)

HEIGHT_COL = "height"

building_clip = gpd.overlay(
    buildings[[HEIGHT_COL, "geometry"]],
    buffers[["origin", "geometry"]],
    how="intersection",
)

building_clip["building_area_m2"] = building_clip.geometry.area

building_metrics = (
    building_clip.groupby("origin", as_index=False)
    .agg(
        building_area_m2=("building_area_m2", "sum"),
        mean_building_height=(HEIGHT_COL, "mean"),
    )
)

buffer_area_m2 = np.pi * BUFFER_M**2

building_metrics["building_density"] = (
    building_metrics["building_area_m2"]
    / buffer_area_m2
)

## 3. Tree-point density

In [ ]:
trees = gpd.read_file(
    DATA_INTERIM / "sevilla_trees_voxcity.gpkg"
)

# The source contains mixed geometry types. For this PoC we use mapped tree points.
tree_points = trees[
    trees.geometry.geom_type == "Point"
].to_crs(25830)

tree_join = gpd.sjoin(
    tree_points[["geometry"]],
    buffers[["origin", "geometry"]],
    how="inner",
    predicate="within",
)

tree_metrics = (
    tree_join.groupby("origin", as_index=False)
    .size()
    .rename(columns={"size": "tree_count"})
)

tree_metrics["tree_density_per_km2"] = (
    tree_metrics["tree_count"]
    / (buffer_area_m2 / 1_000_000)
)

## 4. Mean local UTCI within each buffer

In [ ]:
utci_file = SOLWEIG_DIR / "utci_max.tif"
utci_rows = []

with rasterio.open(utci_file) as src:
    buffers_utci = buffers.to_crs(src.crs)

    for _, row in buffers_utci.iterrows():
        clipped, _ = mask(
            src,
            [mapping(row.geometry)],
            crop=True,
            filled=False,
        )

        values = clipped[0].compressed()

        utci_rows.append({
            "origin": row["origin"],
            "mean_utci_max": (
                values.mean()
                if len(values) > 0
                else np.nan
            ),
        })

utci_metrics = pd.DataFrame(utci_rows)

## 5. Merge local metrics and inspect correlations

In [ ]:
urban_form = (
    origins[
        [
            "origin",
            "normal_access",
            "thermal_access",
            "access_loss_pct",
        ]
    ]
    .merge(building_metrics, on="origin", how="left")
    .merge(tree_metrics, on="origin", how="left")
    .merge(utci_metrics, on="origin", how="left")
)

urban_form[
    ["building_density", "tree_count", "tree_density_per_km2"]
] = urban_form[
    ["building_density", "tree_count", "tree_density_per_km2"]
].fillna(0)

local_variables = [
    "access_loss_pct",
    "building_density",
    "mean_building_height",
    "tree_density_per_km2",
    "mean_utci_max",
]

pearson_local = urban_form[local_variables].corr(method="pearson")
spearman_local = urban_form[local_variables].corr(method="spearman")

local_summary = pd.DataFrame({
    "pearson": pearson_local["access_loss_pct"].drop("access_loss_pct"),
    "spearman": spearman_local["access_loss_pct"].drop("access_loss_pct"),
})

display(urban_form[local_variables].describe())
display(local_summary)

### Early interpretation

The local 250 m morphology variables show weak direct relationships with accessibility loss. Tree-point density is more clearly associated with lower modeled UTCI than with accessibility loss itself, suggesting that accessibility depends on thermal conditions distributed across the reachable network rather than only around the origin.

## 6. Network-scale thermal context

In [ ]:
graph_node_lookup = {
    str(node): node
    for node in G.nodes
}

access["origin_graph"] = access["origin"].map(graph_node_lookup)

TIME_BUDGET = 600


def reachable_network_metrics(graph, origin, budget=600):
    normal_costs = nx.single_source_dijkstra_path_length(
        graph,
        origin,
        cutoff=budget,
        weight="walk_time_s",
    )

    reachable_nodes = set(normal_costs)
    rows = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        if u in reachable_nodes and v in reachable_nodes:
            rows.append({
                "length": float(data.get("length", np.nan)),
                "utci_max": float(data.get("utci_max", np.nan)),
                "heat_excess": float(data.get("heat_excess", np.nan)),
                "cost_ratio": float(data.get("cost_ratio", np.nan)),
            })

    edges_df = pd.DataFrame(rows).dropna(
        subset=["length", "utci_max", "cost_ratio"]
    )

    if edges_df.empty:
        return None

    total_length = edges_df["length"].sum()

    return {
        "reachable_nodes": len(reachable_nodes),
        "reachable_edges": len(edges_df),
        "reachable_length_m": total_length,
        "mean_utci_network": np.average(
            edges_df["utci_max"],
            weights=edges_df["length"],
        ),
        "mean_heat_excess_network": np.average(
            edges_df["heat_excess"],
            weights=edges_df["length"],
        ),
        "mean_cost_ratio": np.average(
            edges_df["cost_ratio"],
            weights=edges_df["length"],
        ),
        "share_length_utci_gt32": (
            edges_df.loc[
                edges_df["utci_max"] > 32,
                "length",
            ].sum()
            / total_length
        ),
        "share_length_utci_gt38": (
            edges_df.loc[
                edges_df["utci_max"] > 38,
                "length",
            ].sum()
            / total_length
        ),
    }


network_rows = []

for origin_str, origin_graph in zip(
    access["origin"],
    access["origin_graph"],
):
    result = reachable_network_metrics(
        G,
        origin_graph,
        budget=TIME_BUDGET,
    )

    if result is not None:
        result["origin"] = origin_str
        network_rows.append(result)

network_metrics = pd.DataFrame(network_rows)

print("Origins processed:", len(network_metrics), "/", len(access))

## 7. Reachable-network contraction

In [ ]:
def reachable_network_change(graph, origin, budget=600):
    normal = nx.single_source_dijkstra_path_length(
        graph,
        origin,
        cutoff=budget,
        weight="walk_time_s",
    )

    thermal = nx.single_source_dijkstra_path_length(
        graph,
        origin,
        cutoff=budget,
        weight="thermal_cost_s",
    )

    normal_nodes = set(normal)
    thermal_nodes = set(thermal)

    return {
        "normal_reachable_nodes": len(normal_nodes),
        "thermal_reachable_nodes": len(thermal_nodes),
        "reachable_node_loss_pct": (
            100
            * (len(normal_nodes) - len(thermal_nodes))
            / len(normal_nodes)
            if normal_nodes
            else np.nan
        ),
    }


contraction_rows = []

for origin_str, origin_graph in zip(
    access["origin"],
    access["origin_graph"],
):
    result = reachable_network_change(
        G,
        origin_graph,
        budget=TIME_BUDGET,
    )
    result["origin"] = origin_str
    contraction_rows.append(result)

contraction = pd.DataFrame(contraction_rows)

network_analysis = (
    access
    .merge(network_metrics, on="origin", how="left")
    .merge(contraction, on="origin", how="left")
)

display(
    network_analysis[
        [
            "access_loss_pct",
            "normal_reachable_nodes",
            "thermal_reachable_nodes",
            "reachable_node_loss_pct",
            "mean_utci_network",
            "mean_cost_ratio",
        ]
    ].describe()
)

In [ ]:
pearson = network_analysis[
    ["reachable_node_loss_pct", "access_loss_pct"]
].corr(method="pearson").iloc[0, 1]

spearman = network_analysis[
    ["reachable_node_loss_pct", "access_loss_pct"]
].corr(method="spearman").iloc[0, 1]

print(f"Pearson r: {pearson:.3f}")
print(f"Spearman rho: {spearman:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    network_analysis["reachable_node_loss_pct"],
    network_analysis["access_loss_pct"],
    alpha=0.75,
)

ax.set_xlabel("Loss of reachable network nodes (%)")
ax.set_ylabel("Accessibility loss (%)")
ax.set_title("Network contraction and accessibility loss")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Interpretation

In this PoC, raw network-node contraction is only weakly associated with transit-stop accessibility loss. This is expected because network nodes are not opportunities: accessibility depends on **which parts of the network disappear and where transit stops are located**. Notebook 12 therefore moves to opportunity-level analysis.